In [2]:
import numpy as np
import pathlib
import trimesh
from scipy.spatial import cKDTree


from beamme.core.mesh import Mesh
from beamme.four_c.element_beam import Beam3rHerm2Line3, Beam3rLine2Line2
from beamme.four_c.material import MaterialReissner
from beamme.core.node import Node
from beamme.four_c.input_file import InputFile
from beamme.mesh_creation_functions.beam_line import create_beam_mesh_line
import beamme.four_c.run_four_c as run_four_c
import pyvista as pv

import sys, os
sys.path.insert(0, os.path.abspath('..'))
from functions.generate_artery import ( generate_straight_artery, generate_curved_artery,
                                        generate_s_bend_artery,
                                        straight_centreline, curved_centreline, s_bend_centreline,
                                        generate_artery_for_stent)
from functions.stent_funcs    import load_update_stent_data
from functions.process_funcs  import map_stent_to_artery, check_stent_artery_fit, downsample_skeleton

### Load stent data & set material parameters

In [ ]:
#  Stent selection 
STENT_NAME = "stent"
#STENT_NAME = "2crownCrimpedXienceStent"
#STENT_NAME = "16crownCrimpedXienceStent+extremaSupports"

# Material parameters 
# Unit system: mm – N – tonne  (consistent with stent geometry in mm)
#   Stress / modulus  →  MPa  =  N / mm²
#   Density           →  kg / m³  (stored for readability; converted to t/mm³ on simulation output)
#
# Preset reference values:
#   Material             E [MPa]    nu [-]  rho [kg/m³]   eps_max [-]
#   Nitinol (NiTi)       50 000     0.33    6 450          0.08   (superelastic)
#   Stainless steel 316L 200 000    0.30    7 900          0.002
#   Cobalt-chromium      230 000    0.30    8 900          0.003

MATERIAL_NAME      = "Nitinol"
YOUNGS_MODULUS     = 50_000.0   # [MPa]   Young's modulus
POISSONS_RATIO     = 0.33       # [-]     Poisson's ratio
DENSITY            = 6_450.0    # [kg/m³] mass density
MAX_ELASTIC_STRAIN = 0.08      # [-]     max recoverable strain before permanent deformation

#  Load stent data 
features, cl_direction, skel = load_update_stent_data(
    STENT_NAME, MATERIAL_NAME, YOUNGS_MODULUS, POISSONS_RATIO,
    DENSITY, MAX_ELASTIC_STRAIN,
)

#  Print summary 
_GEO = {"length", "diameter", "radius", "strut_thickness",
        "z_min", "z_max", "r_inner", "r_outer", "r_mid",
        "center_cylinder_radius", "num_points", "n_regions", "conn_radius_3d"}
_MAT_ORDER = ("material_name", "youngs_modulus", "poissons_ratio", "shear_modulus",
              "density", "max_elastic_strain")

print("\n=== Geometry features ===")
for k, entry in features.items():
    if k in _GEO:
        val   = entry["value"]
        unit  = entry["unit"]
        val_s = f"{val:.6g}" if isinstance(val, float) else str(val)
        print(f"  {k:28s}: {val_s:<16s}  [{unit}]")
print(f"  {'centerline_direction':28s}: {cl_direction.round(6)}  [-]")

print("\n=== Material parameters ===")
for k in _MAT_ORDER:
    entry = features[k]
    val   = entry["value"]
    unit  = entry["unit"]
    val_s = f"{val:.6g}" if isinstance(val, float) else str(val)
    print(f"  {k:28s}: {val_s:<16s}  [{unit}]")

print(f"\nSkeleton nodes : {len(skel):,}")

Loading stent data from: /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/notebook_outputs/stent5
  stent_features.json updated with material parameters.

=== Geometry features ===
  length                      : 15.2302           [mm]
  diameter                    : 1.91844           [mm]
  radius                      : 0.959218          [mm]
  strut_thickness             : 0.113385          [mm]
  z_min                       : -7.61609          [mm]
  z_max                       : 7.61413           [mm]
  r_inner                     : 0.844937          [mm]
  r_outer                     : 0.958322          [mm]
  r_mid                       : 0.90163           [mm]
  center_cylinder_radius      : 0.42283           [mm]
  num_points                  : 1880360           [-]
  n_regions                   : 860               [-]
  conn_radius_3d              : 0.0203409         [mm]
  centerline_direction        : [ 1.0e+00 -1.2e-04 -9.1e-05]  [-]

=== Material parameters ===
  

In [4]:
#  Downsample skeleton (topology-preserving) 
# Thins the straight degree-2 chains between junctions/endpoints by
# DOWNSAMPLE_FACTOR while keeping EVERY junction and endpoint and at least one
# interior node per strut — so graph topology and strut curvature are preserved
# (struts are NOT collapsed to straight lines between junctions).
#
# This reassigns `skel`, so every downstream step (visualisation, mapping,
# compatibility check, and the Step 5 beam mesh) uses the reduced skeleton.
# It is the main lever for keeping the Step 5 beam count — and couple_nodes() —
# tractable. Set DOWNSAMPLE_FACTOR = 1 to disable.
DOWNSAMPLE_FACTOR = 10

skel = downsample_skeleton(skel, DOWNSAMPLE_FACTOR)
print(f"Skeleton nodes after downsampling : {len(skel):,}")

Downsampled skeleton (factor=10): 127,982 → 13,302 nodes, 13,578 edges  (kept 464 junctions/endpoints)
Skeleton nodes after downsampling : 13,302


### Visualise stent skeleton (trimesh — interactive)

In [5]:
# Build unique edge segments from neighbor_ids
coords_idx = skel.set_index("skeleton_point_id")[["x", "y", "z"]]

segments = []
seen = set()
for _, row in skel.iterrows():
    pid = int(row["skeleton_point_id"])
    p0 = coords_idx.loc[pid].to_numpy()
    for nid in row["neighbor_ids"]:
        key = (min(pid, nid), max(pid, nid))
        if key in seen or nid not in coords_idx.index:
            continue
        seen.add(key)
        p1 = coords_idx.loc[nid].to_numpy()
        segments.append([p0, p1])
segments = np.array(segments)
print(f"Unique edges: {len(segments):,}")

# Separate node types for colouring
line_pts     = skel[skel["node_type"] == "line"    ][["x", "y", "z"]].to_numpy()
junction_pts = skel[skel["node_type"] == "junction"][["x", "y", "z"]].to_numpy()

pc_line     = trimesh.PointCloud(line_pts,     colors=np.tile([70,  130, 180, 120], (len(line_pts),     1)))
pc_junction = trimesh.PointCloud(junction_pts, colors=np.tile([255, 140,   0, 255], (len(junction_pts), 1)))

path = trimesh.load_path(segments)
path.colors = np.tile([100, 160, 220, 200], (len(path.entities), 1))

scene = trimesh.Scene([path, pc_line, pc_junction])
scene.show()

Unique edges: 13,578


### Generate matching artery for the stent

In [9]:
#  User parameters 
ARTERY_TYPE     = "s_bend"  # "straight" | "curved" | "s_bend"
NOISE_AMPLITUDE = 0.12 # fractional radius perturbation (0 = smooth, 0.05 = ±5%)
NOISE_SEED      = None         # integer for reproducibility, or None for random
BEND_ANGLE_DEG  = 90      # [deg]  only used for "curved" and "s_bend"

artery_mesh, artery_cl, artery_radius = generate_artery_for_stent(
    features, ARTERY_TYPE, NOISE_AMPLITUDE, NOISE_SEED, BEND_ANGLE_DEG
)

Artery type      : s_bend
Artery radius    : 1.958 mm
Noise amplitude  : 0.12 (12% of radius)  seed=None
Bend angle       : 90.0 deg
Bend radius      : 8.73 mm  (2× arc = 27.41 mm)
Arc length       : 30.46 mm  (stent 15.23 mm = 50% of artery)
Centreline       : 151 points  bounds [0. 0. 0.] → [18.47  0.   19.48]
Mesh             : 9,666 vertices  19,328 faces  watertight=True


### Visualise artery (trimesh — interactive)

In [10]:
artery_vis = artery_mesh.copy()
artery_vis.visual.face_colors = np.tile([210, 100, 90, 180], (len(artery_vis.faces), 1))

# Centreline as a connected gold polyline
cl_segs_vis = np.stack([artery_cl[:-1], artery_cl[1:]], axis=1)
cl_path_vis = trimesh.load_path(cl_segs_vis)
cl_path_vis.colors = np.tile([255, 215, 0, 255], (len(cl_path_vis.entities), 1))

scene_artery = trimesh.Scene([artery_vis, cl_path_vis])
scene_artery.show()

### Map stent skeleton onto artery centreline & combined scene (trimesh — interactive)

In [11]:
#  Map stent skeleton onto artery centreline 
skel_mapped, cl_m = map_stent_to_artery(skel, features, artery_cl)

#  Rebuild edge segments for visualisation 
n_nodes = len(skel_mapped)
segs_m, seen_m = [], set()
for i, nbrs in enumerate(skel["neighbor_ids"]):
    pid = int(skel.iloc[i]["skeleton_point_id"])
    for nid in nbrs:
        if nid >= n_nodes:
            continue
        key = (min(pid, nid), max(pid, nid))
        if key in seen_m:
            continue
        seen_m.add(key)
        segs_m.append([skel_mapped[i], skel_mapped[nid]])
segs_m = np.array(segs_m)

line_mask  = (skel["node_type"] == "line").values
junc_mask  = (skel["node_type"] == "junction").values
line_pts_m = skel_mapped[line_mask]
junc_pts_m = skel_mapped[junc_mask]
print(f"Rebuilt {len(segs_m):,} edge segments")

#  Combined scene 
artery_cloud = trimesh.PointCloud(
    artery_mesh.vertices,
    colors=np.tile([220, 100, 90, 70], (len(artery_mesh.vertices), 1)),
)
cl_path_comb = trimesh.load_path(np.stack([artery_cl[:-1], artery_cl[1:]], axis=1))
cl_path_comb.colors = np.tile([255, 215, 0, 255], (len(cl_path_comb.entities), 1))

path_m    = trimesh.load_path(segs_m)
path_m.colors    = np.tile([100, 160, 220, 220], (len(path_m.entities), 1))
pc_line_m = trimesh.PointCloud(line_pts_m, colors=np.tile([70,  130, 180, 150], (len(line_pts_m), 1)))
pc_junc_m = trimesh.PointCloud(junc_pts_m, colors=np.tile([255, 140,   0, 255], (len(junc_pts_m), 1)))

scene_combined = trimesh.Scene([artery_cloud, cl_path_comb, path_m, pc_line_m, pc_junc_m])
scene_combined.show()

Mapped 13,302 skeleton nodes
Rebuilt 13,578 edge segments


### Geometric & material compatibility check

Before launching the expensive contact simulation, verify that the stent and artery are compatible.

| # | Check | Basis |
|---|---|---|
| 1 | **Length** | Stent arc ≤ 95% of artery arc |
| 2 | **Delivery** | Crimped OD < artery ID |
| 3 | **Containment** | Mapped skeleton nodes inside lumen |
| 4 | **Clearance** | Min wall distance ≥ strut radius |
| 5 | **Bending strain** | ε = r_strut / R_artery < ε_max (material) — or geometric heuristic if ε_max not set |

In [27]:
#  Run 
print("Running compatibility checks …")
print("  [1/5] Length      — inline")
print("  [2/5] Delivery    — inline")
report = check_stent_artery_fit(
    skel_mapped, skel, features, artery_mesh, artery_cl, artery_radius,
)
print("  [5/5] Bending strain — inline")

#  Summary 
_OK   = "\033[92m✓ PASS\033[0m"
_FAIL = "\033[91m✗ FAIL\033[0m"

print()
print("=" * 68)
print("  STENT - ARTERY  COMPATIBILITY REPORT")
print("=" * 68)

for name, r in report.items():
    tag = _OK if r["passed"] else _FAIL
    print(f"\n  {name.upper():<16s}  {tag}")
    print(f"  {'':<16s}  {r['note']}")
    if name == "bending_strain":
        print(f"  {'':<16s}  basis: {r['check_basis']}")
        if r['bending_stiffness_EI_Nmm2'] != "N/A":
            print(f"  {'':<16s}  EI = {r['bending_stiffness_EI_Nmm2']:.4g} N·mm²")

print()
print("=" * 68)
failed = [k for k, v in report.items() if not v["passed"]]
if not failed:
    print("  OVERALL  →  ALL CHECKS PASSED — proceed to simulation")
else:
    print(f"  OVERALL  →  FAILED: {', '.join(f.upper() for f in failed)}")
    print("              Resolve these issues before running simulation.")
print("=" * 68)

Running compatibility checks …
  [1/5] Length      — inline
  [2/5] Delivery    — inline
  [3/5] Containment + [4/5] Clearance … 100.0% inside, min clearance 0.801 mm, 0 penetrating
  [5/5] Bending strain — inline

  STENT - ARTERY  COMPATIBILITY REPORT

  LENGTH            ✓ PASS
                    Stent 2.72 mm fills 66.7% of 4.08 mm artery arc

  DELIVERY          ✓ PASS
                    Crimped OD 1.164 mm < artery ID 3.164 mm (63.2% radial margin)

  CONTAINMENT       ✓ PASS
                    100.0% of 2,015 sampled nodes inside artery lumen

  CLEARANCE         ✓ PASS
                    Min wall clearance 0.801 mm (strut radius 0.078 mm), 0 penetrating nodes

  BENDING_STRAIN    ✓ PASS
                    Max strut bending strain 3.316% at R_min=2.3 mm (< elastic limit 8.0%)
                    basis: material  [ε = r_strut/R,  limit = 8.0%]
                    EI = 1.422 N·mm²

  OVERALL  →  ALL CHECKS PASSED — proceed to simulation


### Step 5 — 4C contact simulation input

Converts the placed stent skeleton and artery mesh into a 4C `.dat` input file for beam-to-solid contact simulation.

| Entity | Discretisation | Treatment |
|--------|---------------|-----------|
| Stent struts | `BEAM3R LINE2` (Reissner beam) | elastic, free to expand |
| Artery wall | `SHELL_REISSNER TRI3` | rigid first pass — all DOFs fixed |
| Contact | beam-to-solid, penalty method | stent = slave · artery = master |

Set the parameters in the next cell, then run the generator. The expansion load section is left as a documented TODO in the output file — choose the deployment strategy with your supervisor before running 4C.

In [28]:
# ═══════════════════════════════════════════════════════════════════════
# Step 5 — 4C contact simulation input via BeamMe
# ═══════════════════════════════════════════════════════════════════════


#  Simulation parameters (Ranno et al. 2025) 
RADIAL_LINE_LOAD = 0.01e-3
N_LOAD_STEPS     = 100
CONTACT_PENALTY  = 1.0e5
ARTERY_THICKNESS = 0.5

_E   = features["youngs_modulus"]["value"]
_nu  = features["poissons_ratio"]["value"]
_rho = features["density"]["value"] * 1e-12
_r_s = features["strut_thickness"]["value"] / 2

#  1. Mesh & material 
mesh = Mesh()
stent_material = MaterialReissner(radius=_r_s, youngs_modulus=_E, nu=_nu, density=_rho)

#  2. Beam elements from skeleton edges 
coords_lookup = {int(row["skeleton_point_id"]): skel_mapped[i]
                 for i, row in skel.iterrows()}
edges = {(min(pid, nid), max(pid, nid))
         for _, row in skel.iterrows()
         for pid in [int(row["skeleton_point_id"])]
         for nid in row["neighbor_ids"] if nid in coords_lookup}
edges = list(edges)
print(f"Creating {len(edges):,} beam elements from skeleton edges")

# Note: Beam3rHerm2Line3 has 3 nodes per element (2x endpoints + 1x mid-node) and a cubic Hermite interpolation,
# which can capture bending with fewer elements than the linear Beam3rLine2Line2 — but it also has more DOFs per element and may be less stable for very large deformations.
# Note: Beam3rLine2Line2 is a simpler linear beam element with 2 nodes per element, 

for n1, n2 in edges:
    create_beam_mesh_line(mesh=mesh, beam_class=Beam3rHerm2Line3, # beam_class = Beam3rHerm2Line3 or Beam3rLine2Line2
                          material=stent_material,
                          start_point=coords_lookup[n1], end_point=coords_lookup[n2], n_el=1)

mesh.couple_nodes(reuse_matching_nodes=True)
print(f"Mesh: {len(mesh.nodes):,} nodes, {len(mesh.elements):,} elements")

#  3. Visualise beam mesh 
# Adjust these to taste:
SHOW_DIRECTORS = True   # cross-section orientation arrows
DIRECTOR_SCALE = 1    # arrow length as × strut radius  (BeamMe default = 3.5)
NODE_SCALE     = 0.5    # sphere size as × strut radius   (BeamMe default = 1.5)
N_SIDES        = 5     # tube / sphere resolution

vtk_beam, _ = mesh.get_vtk_representation()

# BeamMe pipeline: cell_data_to_point_data → extract_surface → tube
# (tube() is only available on PolyData, not UnstructuredGrid)
beam_grid    = pv.UnstructuredGrid(vtk_beam.grid).cell_data_to_point_data()
beam_surface = beam_grid.extract_surface()
tube         = beam_surface.tube(scalars="cross_section_radius", absolute=True, n_sides=N_SIDES)

# Node points (node_value ≈ 1 → end node, < 0.5 → mid node)
node_cloud   = beam_grid.cast_to_poly_points().threshold(scalars="node_value", value=(0.4, 1.1))
sphere       = pv.Sphere(radius=1.0, theta_resolution=N_SIDES, phi_resolution=N_SIDES)
end_nodes    = node_cloud.threshold(scalars="node_value", value=(0.9, 1.1))
mid_nodes    = node_cloud.threshold(scalars="node_value", value=(0.4, 0.6))

pl = pv.Plotter(notebook=True)
pl.set_background("White")
pl.add_axes()

# pl.add_mesh(tube, color="#4a90d9", smooth_shading=True)

if end_nodes.n_points > 0:
    end_nodes["r"] = np.full(end_nodes.n_points, _r_s * NODE_SCALE)
    pl.add_mesh(end_nodes.glyph(geom=sphere, scale="r", factor=1.0, orient=False),
                color="#2ecc71", smooth_shading=True)
if mid_nodes.n_points > 0:
    mid_nodes["r"] = np.full(mid_nodes.n_points, _r_s * NODE_SCALE)
    pl.add_mesh(mid_nodes.glyph(geom=sphere, scale="r", factor=1.0, orient=False),
                color="#1abc9c", smooth_shading=True)

if SHOW_DIRECTORS:
    arrow  = pv.Arrow(tip_length=0.25, tip_radius=0.08, shaft_radius=0.03)
    colors = ["#e74c3c", "#27ae60", "#2980b9"]
    for key, col in zip(["base_vector_1", "base_vector_2", "base_vector_3"], colors):
        dir_cloud      = beam_grid.cast_to_poly_points()
        dir_cloud["r"] = np.full(dir_cloud.n_points, _r_s * DIRECTOR_SCALE)
        pl.add_mesh(dir_cloud.glyph(geom=arrow, orient=key, scale="r", factor=1.0),
                    color=col, opacity=0.85)

pl.show()


Creating 2,033 beam elements from skeleton edges
Mesh: 6,099 nodes, 2,033 elements


/var/folders/pm/492pcbb56nl8wv37h9lh6w3w0000gn/T/ipykernel_45028/2990807395.py:55: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  beam_surface = beam_grid.extract_surface()


Widget(value='<iframe src="http://localhost:56672/index.html?ui=P_0x3be0cb250_1&reconnect=auto" class="pyvista…